# Starter 4: Interference and the phase you cannot see

| | |
|---|---|
| **Level** | Introductory |
| **Time** | About 30 minutes |
| **Prerequisites** | Qubits, the Hadamard gate, phase |
| **Default device** | IQM Garnet |
| **Also runs on** | Rigetti Cepheus-1-108Q, AQT IBEX Q1 |
| **Qubits** | 12 |
| **Two-qubit gates** | None |
| **Hardware jobs** | 1 |
| **Approximate cost** | Garnet at 500 shots: about 103 credits. Rigetti: about 10 credits (billed by execution time). |
| **Suggested hand-in** | Your plot, the fitted visibility, and answers to Questions 1 and 2 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST starter series from qBraid. You may copy, edit and adapt this notebook for your course.*

The gate $R_z(\varphi)$ changes the relative phase between $|0\rangle$ and $|1\rangle$. Measured directly, a phase has no effect: a qubit in an equal superposition gives 0 or 1 with probability one half, whatever $\varphi$ is.

A phase becomes visible through interference. The sequence H, $R_z(\varphi)$, H splits the state into two paths, shifts the phase of one, and recombines them. The probability of measuring 0 is then

$$P(0) = \cos^2(\varphi/2),$$

which swings between 1 and 0 as $\varphi$ changes. This is a one-qubit version of a Mach-Zehnder interferometer. The two Hadamard gates play the role of the beam splitters.

In this notebook you measure this interference fringe, together with three control qubits that skip the second Hadamard and so should show no dependence on $\varphi$.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "aws:iqm:qpu:garnet"   # device list and prices: see the README
SHOTS = 500                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuit

Qubits 0 to 8 run the interference sequence at nine phases between 0 and $2\pi$. Qubits 9 to 11 are the controls: H and $R_z(\varphi)$, but no second H.

In [ ]:
phis = np.linspace(0, 2 * np.pi, 9)
control_phis = [0, np.pi / 2, np.pi]
N_QUBITS = len(phis) + len(control_phis)

qc = QuantumCircuit(N_QUBITS)
for qubit, phi in enumerate(phis):                  # interference: H, Rz, H
    qc.h(qubit)
    qc.rz(phi, qubit)
    qc.h(qubit)
for k, phi in enumerate(control_phis):              # controls: H, Rz, no second H
    qubit = len(phis) + k
    qc.h(qubit)
    qc.rz(phi, qubit)
qc.measure_all()

qc.draw(output="text", fold=100)

## 2. Ideal simulation

In [ ]:
simulator = AerSimulator()
ideal_counts = simulator.run(qc, shots=SHOTS).result().get_counts()

def fringe(counts):
    return [1 - prob_one(counts, q, N_QUBITS) for q in range(len(phis))]

def controls(counts):
    return [1 - prob_one(counts, len(phis) + k, N_QUBITS) for k in range(len(control_phis))]

ideal_fringe = fringe(ideal_counts)
print("interference qubits, P(0):", np.round(ideal_fringe, 3))
print("control qubits, P(0):     ", np.round(controls(ideal_counts), 3))

## 3. Run on hardware

In [ ]:
N_JOBS = 1

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    job = device.run(qc, shots=SHOTS)
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = job.result().data.get_counts()
    print(f"Received {sum(hw_counts.values())} shots.")
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Compare

The **visibility** of a fringe measures how strongly it swings. Fitting $P(0) = a + b\cos\varphi$, the visibility is $V = b/a$. A perfect interferometer has $V = 1$.

In [ ]:
def visibility(p0):
    basis = np.column_stack([np.ones_like(phis), np.cos(phis)])
    (a, b), *_ = np.linalg.lstsq(basis, np.array(p0), rcond=None)
    return b / a

fine = np.linspace(0, 2 * np.pi, 200)
plt.plot(fine, np.cos(fine / 2) ** 2, color="gray", label="prediction")
plt.plot(phis, ideal_fringe, "o", color="black", label="ideal simulation")
print(f"ideal simulation: visibility = {visibility(ideal_fringe):.3f}")

if hw_counts:
    hw_fringe = fringe(hw_counts)
    print(f"hardware:         visibility = {visibility(hw_fringe):.3f}")
    print(f"hardware control qubits, P(0): {np.round(controls(hw_counts), 3)}")
    plt.plot(phis, hw_fringe, "s", color="tab:orange", label=f"hardware ({DEVICE_ID})")
    plt.plot(control_phis, controls(hw_counts), "^", color="tab:green", label="hardware, control qubits")

plt.xlabel("phase phi")
plt.ylabel("P(0)")
plt.legend()
plt.show()

## Questions to try

1. What is the visibility on hardware? List two physical effects that could reduce it.
2. Why do the control qubits give $P(0) \approx 0.5$ for every phase? What does that tell you about whether a phase can be measured directly?
3. Dephasing scrambles the phase of a superposition without changing the populations of $|0\rangle$ and $|1\rangle$. How would dephasing affect the interference qubits? How would it affect the control qubits?
4. Replace $R_z(\varphi)$ with the phase gate $P(\varphi)$ (`qc.p(phi, qubit)`). Does anything change? Why or why not?